In [0]:
%run ./Config

In [0]:

df_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", bootstrap) \
    .option("kafka.security.protocol", "SASL_SSL") \
    .option("kafka.sasl.mechanism", "PLAIN") \
    .option("kafka.sasl.jaas.config", f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{api_key}" password="{api_secret}";') \
    .option("subscribe", "topic_0") \
    .option("startingOffsets", "earliest") \
    .load()

#bootstrap : address of your Confluent Kafka broker
# SASL — how you authenticate (username/password)
# SSL — the connection is encrypted
# PLAIN means username + password authentication
#JAAS Config : This is the actual credential string. 
#JAAS (Java Authentication and Authorization Service) is Java's standard way of passing auth config.

print("Stream created successfully!")

In [0]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("product", StringType()),
    StructField("amount", DoubleType())
])

df_parsed = df_stream \
    .selectExpr("CAST(value AS STRING) as json_str") \
    .select(from_json(col("json_str"), schema).alias("data")) \
    .select("data.*")

query = df_parsed.writeStream \
    .format("delta") \
    .option("checkpointLocation", "dbfs:/Workspace/Users/hemasundher0000@gmail.com/kafka_checkpoint") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable("orders_streaming")

query.awaitTermination()
print("Done!")

In [0]:
spark.sql("SELECT * FROM orders_streaming").show()

In [0]:
import os
print(os.environ.get('HOME'))

In [0]:
display(dbutils.fs.ls("dbfs:/Workspace/"))